# EcoCode SDK Jupyter Notebook Demo

This notebook demonstrates how to track energy consumption, carbon footprint, and training/inference metrics using the EcoCode Python SDK in **Standalone Mode** during real machine learning workflows on the GPU.

In [1]:
# Environment verification and configuration
import os
import sys
import time

# Auto-configure CUDA_HOME with local mock if developer CUDA Toolkit is missing (DeepSpeed workaround)
if "CUDA_HOME" not in os.environ:
    possible_mock = os.path.abspath(os.path.join(os.getcwd(), "venv", "cuda_mock"))
    if os.path.exists(possible_mock):
        os.environ["CUDA_HOME"] = possible_mock
        print(f"Set CUDA_HOME to mock toolkit path: {possible_mock}")

import torch
import ecocode

# Configure API URL to point to your local EcoCode backend
os.environ.setdefault("ECOCODE_API_URL", "http://localhost:3001")

api_key = os.environ.get("ECOCODE_API_KEY", "ecocode_fhAIugP2IeEDrRYiKD5Fl3exDWEGCIe0")

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
print(f"Backend URL: {os.environ['ECOCODE_API_URL']}")
print(f"API Key: {api_key[:8]}...")

CUDA Available: True
Device Name: NVIDIA GeForce RTX 4090
Backend URL: http://localhost:3001
API Key: ecocode_...


## Method 1: Context Manager (`with` block)
We will train a PyTorch neural network on a synthetic dataset on the GPU, wrapping the entire loop in an EcoCode context manager. To ensure we collect a substantial amount of carbon tracking data, we run this for 10 epochs and simulate a realistic batch computation latency.

In [2]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Define a simple model and synthetic dataset
class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(100, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    def forward(self, x):
        return self.net(x)

# Synthetic data: 10,000 samples
X = torch.randn(10000, 100)
y = torch.randint(0, 10, (10000,))
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Starting training on {device}...")

# Wrap training loop with EcoCode
with ecocode.run("notebook-pytorch-training", api_key=api_key, interval=1.0) as run:
    run.mark("phase", "data_loading")
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    run.mark("phase", "training")
    for epoch in range(10): # 10 epochs
        run.log_epoch(epoch)
        total_loss = 0.0
        
        for step, (inputs, targets) in enumerate(loader):
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            # Simulate realistic batch processing time to allow energy metrics to accumulate
            time.sleep(0.04)
            
            # Log metrics at interval
            if step % 10 == 0:
                global_step = epoch * len(loader) + step
                run.log_step(global_step)
                run.log_loss(loss.item())
                run.log_lr(optimizer.param_groups[0]['lr'])
                
        print(f"Epoch {epoch+1}/10 Completed | Avg Loss: {total_loss/len(loader):.4f}")
        
    run.mark("phase", "done")

print("Training finished successfully and metrics reported!")

Starting training on cuda...


/home/ai-lab1/Desktop/aziz/EcoCode/python-sdk/venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: UserWarning: [ecocode] Monitor binary resolved from an unexpected path: '/home/ai-lab1/.cargo/bin/ecocode'. If you did not install EcoCode here, check your PATH for unexpected entries.
  exec(code_obj, self.user_global_ns, self.user_ns)


Epoch 1/10 Completed | Avg Loss: 2.3318
Epoch 2/10 Completed | Avg Loss: 2.2505
Epoch 3/10 Completed | Avg Loss: 2.1950
Epoch 4/10 Completed | Avg Loss: 2.1282
Epoch 5/10 Completed | Avg Loss: 2.0372
Epoch 6/10 Completed | Avg Loss: 1.9283
Epoch 7/10 Completed | Avg Loss: 1.8004
Epoch 8/10 Completed | Avg Loss: 1.6596
Epoch 9/10 Completed | Avg Loss: 1.5195
Epoch 10/10 Completed | Avg Loss: 1.3908
Training finished successfully and metrics reported!


## Method 2: Hugging Face Trainer Integration with `EcoCodeCallback`
Next, we show a realistic fine-tuning task using Hugging Face `transformers` and `datasets`. We will fine-tune a small model and use the pre-built `EcoCodeCallback` to automatically stream performance metrics, epochs, and carbon footprint. We will run it for 3 epochs with a larger synthetic dataset to let it run for a longer duration.

In [3]:
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from datasets import Dataset
from ecocode.integrations.huggingface import EcoCodeCallback

# Create a larger synthetic text dataset (700 samples)
texts = [
    "Machine learning carbon footprint tracking is essential.",
    "EcoCode helps developers measure training energy consumption.",
    "Green AI aims to optimize large language model efficiency.",
    "Monitoring GPU power usage is key to sustainable deep learning.",
    "Sustainable software engineering reduces operational emissions.",
    "Training large deep learning models is highly resource intensive.",
    "Using energy-efficient hardware mitigates environmental impact."
] * 150
dataset_dict = {"text": texts}
hf_dataset = Dataset.from_dict(dataset_dict)

# Load a small tokenizer and model (GPT-2)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_func(examples):
    return tokenizer(examples["text"], truncation=True, max_length=64, padding="max_length")

tokenized_ds = hf_dataset.map(tokenize_func, batched=True, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Load tiny model
hf_model = AutoModelForCausalLM.from_pretrained("gpt2")
hf_model.to(device)

print("Starting Hugging Face training run...")

# Run training inside EcoCode context
with ecocode.run("notebook-hf-integration", api_key=api_key, interval=1.0) as eco_run:
    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=3, # 3 epochs
        per_device_train_batch_size=8,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        fp16=(device.type == "cuda"),
        push_to_hub=False
    )
    
    class SlowEcoCodeCallback(EcoCodeCallback):
        def on_step_end(self, args, state, control, **kwargs):
            time.sleep(0.15)
            
    trainer = Trainer(
        model=hf_model,
        args=training_args,
        train_dataset=tokenized_ds,
        data_collator=collator,
        callbacks=[SlowEcoCodeCallback(eco_run)],
    )
    
    trainer.train()

print("Hugging Face training completed!")

/home/ai-lab1/Desktop/aziz/EcoCode/python-sdk/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███| 148/148 [00:00<00:00, 13959.95it/s]


Starting Hugging Face training run...


/home/ai-lab1/Desktop/aziz/EcoCode/python-sdk/venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: UserWarning: [ecocode] Monitor binary resolved from an unexpected path: '/home/ai-lab1/.cargo/bin/ecocode'. If you did not install EcoCode here, check your PATH for unexpected entries.
  exec(code_obj, self.user_global_ns, self.user_ns)
/home/ai-lab1/miniconda3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/ai-lab1/miniconda3/compiler_compat/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.129763
20,0.557081
30,0.247205
40,0.217491
50,0.239010
60,0.198572
70,0.173414
80,0.156035
90,0.172376
100,0.164281


Hugging Face training completed!


## Method 3: Explicit Start / Stop API (Interactive Modeling)
In Jupyter notebooks, you might want to start a tracking session in one cell, experiment or validate interactively across multiple cells, and stop tracking in a final cell.

In [4]:
# Initialize and start the run
interactive_run = ecocode.run("notebook-interactive-experiment", api_key=api_key, interval=1.0)
interactive_run.start()
print(f"Tracking session started. Run ID: {interactive_run._run_id}")

/tmp/ipykernel_1200517/2203636354.py:3: UserWarning: [ecocode] Monitor binary resolved from an unexpected path: '/home/ai-lab1/.cargo/bin/ecocode'. If you did not install EcoCode here, check your PATH for unexpected entries.
  interactive_run.start()


Tracking session started. Run ID: 50


In [5]:
# Perform evaluation or interactive inference
interactive_run.mark("phase", "evaluation")
print("Running validation inference...")

# Run model evaluation
hf_model.eval()
test_input = tokenizer("Green AI", return_tensors="pt").to(device)
with torch.no_grad():
    # Simulate a series of 50 interactive inference steps
    for _ in range(50):
        outputs = hf_model(**test_input)
        time.sleep(0.2)
    
interactive_run.log_accuracy(98.5)
interactive_run.log_custom("val_loss", float(outputs.loss or 0.0))
print("Evaluation step done.")

Running validation inference...
Evaluation step done.


In [6]:
# Finalize the run and stop monitoring
interactive_run.stop()
print("Tracking session finalized and stopped.")

Tracking session finalized and stopped.
